# BTC.B Vault — Withdrawal Waterfall Model

**Strategy Overview:**
1. Users deposit BTC.B into the vault
2. Vault borrows USDC/USDT on Morpho at 65% LTV
3. Borrowed capital is allocated to a basket of RWA assets

**Constraints:**
- BTC loan LTV must **never exceed 70%**
- SLA: honor at least **50% of withdrawals within 7 days**

**Withdrawal Waterfall Priority:**
1. Deplete cash balances to repay loan
2. Redeem shorter-term assets (≤7 day liquidity)
3. Collateralize longer-term assets and borrow to front withdrawals

In [ ]:
import pandas as pd
import numpy as np
# ─────────────────────────────────────────────
# 1. VAULT PARAMETERS
# ─────────────────────────────────────────────
INITIAL_AUM        = 100_000_000   # BTC.B collateral value in USD
TARGET_LTV         = 0.65          # Initial borrow LTV
MAX_LTV            = 0.70          # Hard ceiling – must never exceed
BORROW_RATE        = 0.036         # Annual Morpho borrow rate on BTC.B loan
RWA_YIELD          = 0.07744       # Weighted avg RWA yield (from sheet)
SECONDARY_BORROW_RATE = 0.08       # Conservative rate when borrowing against illiquid RWA

In [ ]:
# ─────────────────────────────────────────────
# 2. RWA BASKET  (from Excel – Sheet1)
# ─────────────────────────────────────────────
assets = pd.DataFrame({
    "Asset"            : ["Morpho USDC (Cash)", "PRIME", "STAC", "PST",
                          "sNUSD", "sUSDai", "FalconX AA", "ONyc"],
    "Target Weight"    : [0.10, 0.25, 0.15, 0.15, 0.15, 0.10, 0.05, 0.05],
    "Days to Liquidity": [0,    1,    5,    7,    10,   30,   60,   90  ],
    "LLTV"             : [0.00, 0.91, 0.00, 0.80, 0.91, 0.91, 0.77, 0.70],
    "Yield"            : [0.04, 0.085,0.0468,0.10,0.0954,0.0724,0.095,0.1025],
    "MM Liquidity ($M)": [0,   14.8,  0,   1.15, 0.93, 0.001, 14.95, 11.05],
})

# Borrowed capital and initial allocation
borrowed_capital = INITIAL_AUM * TARGET_LTV          # $65 M
assets["Allocation ($)"] = assets["Target Weight"] * borrowed_capital

# Can this asset be collateralised for emergency liquidity?
assets["Can Collateralise"] = assets["LLTV"] > 0

print("=" * 65)
print("  INITIAL VAULT STATE")
print("=" * 65)
print(f"  BTC.B Collateral Value : ${INITIAL_AUM:>15,.0f}")
print(f"  Borrowed Capital (USDC): ${borrowed_capital:>15,.0f}")
print(f"  Starting LTV           : {TARGET_LTV:.1%}")
print(f"  Max Allowed LTV        : {MAX_LTV:.1%}")
print()
print("  RWA Basket Allocation:")
print(assets[["Asset","Target Weight","Allocation ($)",
              "Days to Liquidity","LLTV","Yield","MM Liquidity ($M)","Can Collateralise"]].to_string(index=False))
print(f"\n  Total Allocated        : ${assets['Allocation ($)'].sum():>15,.0f}")
print(f"  Weighted Avg Yield     : {(assets['Target Weight'] * assets['Yield']).sum():.2%}")

In [ ]:
# ─────────────────────────────────────────────
# 3. CORE WATERFALL FUNCTION
# ─────────────────────────────────────────────
def run_waterfall(withdrawal_pct: float,
                  assets_df: pd.DataFrame,
                  initial_aum: float   = INITIAL_AUM,
                  borrowed:    float   = None,
                  target_ltv:  float   = TARGET_LTV,
                  max_ltv:     float   = MAX_LTV,
                  sec_borrow_rate: float = SECONDARY_BORROW_RATE,
                  verbose: bool = False) -> dict:
    """
    Simulates a vault withdrawal waterfall.

    Parameters
    ----------
    withdrawal_pct  : fraction of AUM that users wish to withdraw (0.05 – 0.75)
    assets_df       : RWA basket dataframe with current allocations

    Returns
    -------
    dict with full scenario metrics
    """
    if borrowed is None:
        borrowed = initial_aum * target_ltv

    # Work on a copy so we don't mutate the master frame
    basket = assets_df.copy().sort_values("Days to Liquidity").reset_index(drop=True)

    withdrawal_usd   = initial_aum * withdrawal_pct   # gross user request
    loan_to_repay    = borrowed * withdrawal_pct       # proportional loan payback
    loan_outstanding = borrowed                        # BTC.B-backed Morpho loan
    collateral_value = initial_aum                     # BTC.B collateral

    # Post-withdrawal: release BTC.B, reduce collateral
    collateral_value -= withdrawal_usd

    # ── helpers ──────────────────────────────
    def current_ltv():
        return loan_outstanding / collateral_value if collateral_value > 0 else 999

    def ltv_ok():
        return current_ltv() <= max_ltv

    # ── tracking ─────────────────────────────
    log          = []
    total_repaid = 0.0          # cumulative loan repaid from RWA proceeds
    liquidity_raised_7d = 0.0   # tracks how much raised within ≤7 days
    secondary_borrows = []      # (asset, amount_borrowed, rate)
    remaining_to_repay = loan_to_repay   # still need to source USDC to repay

    # ════════════════════════════════════════════════════════
    # STEP 1 – Cash first  (Days to Liquidity == 0)
    # ════════════════════════════════════════════════════════
    for idx, row in basket.iterrows():
        if row["Days to Liquidity"] > 0:
            continue
        available = row["Allocation ($)"]
        used      = min(available, remaining_to_repay)
        basket.at[idx, "Allocation ($)"] -= used

        # Cash goes straight to repay loan
        loan_outstanding -= used
        total_repaid     += used
        remaining_to_repay -= used
        liquidity_raised_7d  += used

        log.append({
            "Step"     : "1 - Cash",
            "Asset"    : row["Asset"],
            "Action"   : "Redeem & repay loan",
            "Amount"   : used,
            "Days"     : row["Days to Liquidity"],
            "LTV After": current_ltv(),
        })
        if remaining_to_repay <= 0:
            break

    # ════════════════════════════════════════════════════════
    # STEP 2 – Redeem short-term assets  (1 <= Days <= 7)
    # ════════════════════════════════════════════════════════
    if remaining_to_repay > 0:
        for idx, row in basket.iterrows():
            if not (1 <= row["Days to Liquidity"] <= 7):
                continue
            available = row["Allocation ($)"]
            used      = min(available, remaining_to_repay)
            basket.at[idx, "Allocation ($)"] -= used

            loan_outstanding -= used
            total_repaid     += used
            remaining_to_repay -= used
            liquidity_raised_7d  += used

            log.append({
                "Step"     : "2 - Short-term redemption (<=7d)",
                "Asset"    : row["Asset"],
                "Action"   : "Redeem & repay loan",
                "Amount"   : used,
                "Days"     : row["Days to Liquidity"],
                "LTV After": current_ltv(),
            })
            if remaining_to_repay <= 0:
                break

    # ════════════════════════════════════════════════════════
    # STEP 3 – Collateralise longer-dated assets to borrow
    #           Only for assets with LLTV > 0
    #           Borrow at 90% of LLTV for safety headroom
    #           Capped by on-chain MM liquidity
    # ════════════════════════════════════════════════════════
    if remaining_to_repay > 0:
        for idx, row in basket.iterrows():
            if row["Days to Liquidity"] <= 7:
                continue
            if not row["Can Collateralise"]:
                continue
            available = row["Allocation ($)"]
            if available <= 0:
                continue

            # Borrow at 90% of LLTV for safety headroom
            borrow_factor = row["LLTV"] * 0.90
            max_borrow_collateral = available * borrow_factor
            # Also cap by on-chain MM liquidity
            mm_liq = row["MM Liquidity ($M)"] * 1_000_000
            max_borrow = min(max_borrow_collateral, mm_liq) if mm_liq > 0 else max_borrow_collateral
            borrow_now = min(max_borrow, remaining_to_repay)

            # This borrow repays the Morpho loan
            loan_outstanding -= borrow_now
            total_repaid     += borrow_now
            remaining_to_repay -= borrow_now

            secondary_borrows.append({
                "Asset"           : row["Asset"],
                "Collateral Value": available,
                "Borrow Amount"   : borrow_now,
                "Borrow Rate"     : sec_borrow_rate,
                "Days to Unwind"  : row["Days to Liquidity"],
                "MM Liq Cap ($M)" : row["MM Liquidity ($M)"],
            })
            log.append({
                "Step"     : "3 - Collateralise long-dated",
                "Asset"    : row["Asset"],
                "Action"   : f"Pledge collateral, borrow {borrow_factor:.0%}",
                "Amount"   : borrow_now,
                "Days"     : row["Days to Liquidity"],
                "LTV After": current_ltv(),
            })
            if remaining_to_repay <= 0:
                break

    # ════════════════════════════════════════════════════════
    # FINAL METRICS
    # ════════════════════════════════════════════════════════
    total_secondary_borrowed = sum(s["Borrow Amount"] for s in secondary_borrows)
    shortfall                = max(remaining_to_repay, 0)

    # 7-day SLA: cash + short-term + secondary borrows (on-chain = instant)
    seven_day_liquidity   = liquidity_raised_7d + total_secondary_borrowed
    sla_pct               = seven_day_liquidity / loan_to_repay if loan_to_repay > 0 else 1.0
    sla_met               = sla_pct >= 0.50

    # Annual cost of secondary borrows
    annual_secondary_cost = sum(s["Borrow Amount"] * s["Borrow Rate"]
                                for s in secondary_borrows)

    # Remaining RWA portfolio yield
    remaining_gross_yield = (basket["Allocation ($)"] * basket["Yield"]).sum()
    remaining_morpho_cost = loan_outstanding * BORROW_RATE
    net_yield             = remaining_gross_yield - remaining_morpho_cost - annual_secondary_cost

    # Break-even BTC drop before LTV hits 70%
    post_ltv = current_ltv()
    breakeven_btc_drop = 1 - (post_ltv / max_ltv) if post_ltv < max_ltv else 0

    result = {
        "Withdrawal %"                 : withdrawal_pct,
        "Withdrawal ($)"               : withdrawal_usd,
        "Loan to Repay ($)"            : loan_to_repay,
        "Loan Repaid ($)"              : total_repaid,
        "Loan Outstanding ($)"         : loan_outstanding,
        "Collateral Value ($)"         : collateral_value,
        "Primary LTV After"            : post_ltv,
        "Primary LTV Healthy"          : post_ltv <= max_ltv,
        "Secondary Borrows ($)"        : total_secondary_borrowed,
        "Shortfall ($)"                : shortfall,
        "7d Liquidity ($)"             : seven_day_liquidity,
        "7d SLA %"                     : sla_pct,
        "50% SLA Met"                  : sla_met,
        "Annual Secondary Borrow Cost" : annual_secondary_cost,
        "Remaining Gross Yield ($)"    : remaining_gross_yield,
        "Remaining Morpho Cost ($)"    : remaining_morpho_cost,
        "Net Yield ($)"                : net_yield,
        "Net Yield on Remaining AUM"   : net_yield / collateral_value if collateral_value > 0 else 0,
        "Break-even BTC Drop"          : breakeven_btc_drop,
        "Waterfall Log"                : log,
        "Secondary Borrow Detail"      : secondary_borrows,
        "Basket After"                 : basket,
    }
    return result

In [ ]:
# ─────────────────────────────────────────────
# 4. RUN ALL SCENARIOS
# ─────────────────────────────────────────────
scenarios = [0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.60, 0.75]
results   = {pct: run_waterfall(pct, assets) for pct in scenarios}

In [ ]:
# ─────────────────────────────────────────────
# 5. SUMMARY TABLE
# ─────────────────────────────────────────────
summary_rows = []
for pct, r in results.items():
    summary_rows.append({
        "Withdrawal %"              : f"{pct:.0%}",
        "Withdrawal ($M)"           : f"${r['Withdrawal ($)']/1e6:.1f}M",
        "Loan to Repay ($M)"        : f"${r['Loan to Repay ($)']/1e6:.1f}M",
        "Loan Repaid ($M)"          : f"${r['Loan Repaid ($)']/1e6:.1f}M",
        "Loan Outstanding ($M)"     : f"${r['Loan Outstanding ($)']/1e6:.1f}M",
        "Primary LTV"               : f"{r['Primary LTV After']:.2%}",
        "LTV OK"                    : "Y" if r["Primary LTV Healthy"] else "N",
        "Secondary Borrows ($M)"    : f"${r['Secondary Borrows ($)']/1e6:.2f}M",
        "Shortfall ($M)"            : f"${r['Shortfall ($)']/1e6:.2f}M",
        "7d Liquidity ($M)"         : f"${r['7d Liquidity ($)']/1e6:.1f}M",
        "SLA Met"                   : "Y" if r["50% SLA Met"] else "N",
    })

summary_df = pd.DataFrame(summary_rows)
print("=" * 130)
print("  WITHDRAWAL WATERFALL - SCENARIO SUMMARY")
print("=" * 130)
print(summary_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# 6. DETAILED WATERFALL LOG (example: 50% withdrawal)
# ─────────────────────────────────────────────
example_pct = 0.50
r = results[example_pct]

print(f"{'='*80}")
print(f"  DETAILED WATERFALL: {example_pct:.0%} WITHDRAWAL (${r['Withdrawal ($)']/1e6:.1f}M)")
print(f"{'='*80}")
print(f"  Loan to Repay: ${r['Loan to Repay ($)']/1e6:.2f}M")
print()

log_df = pd.DataFrame(r["Waterfall Log"])
log_df["Amount"] = log_df["Amount"].apply(lambda x: f"${x/1e6:.3f}M")
log_df["LTV After"] = log_df["LTV After"].apply(lambda x: f"{x:.2%}")
print(log_df.to_string(index=False))

if r["Secondary Borrow Detail"]:
    print(f"\n  Secondary Borrow Detail:")
    sec_df = pd.DataFrame(r["Secondary Borrow Detail"])
    sec_df["Collateral Value"] = sec_df["Collateral Value"].apply(lambda x: f"${x/1e6:.2f}M")
    sec_df["Borrow Amount"] = sec_df["Borrow Amount"].apply(lambda x: f"${x/1e6:.3f}M")
    sec_df["Borrow Rate"] = sec_df["Borrow Rate"].apply(lambda x: f"{x:.1%}")
    print(sec_df.to_string(index=False))

print(f"\n  --- Final State ---")
print(f"  Loan Outstanding   : ${r['Loan Outstanding ($)']/1e6:.2f}M")
print(f"  Collateral Value   : ${r['Collateral Value ($)']/1e6:.1f}M")
print(f"  Primary LTV        : {r['Primary LTV After']:.2%}")
print(f"  LTV Healthy        : {'YES' if r['Primary LTV Healthy'] else 'NO'}")
print(f"  Shortfall          : ${r['Shortfall ($)']/1e6:.3f}M")
print(f"  7d SLA Coverage    : {r['7d SLA %']:.1%}")
print(f"  50% SLA Met        : {'YES' if r['50% SLA Met'] else 'NO'}")

---
## Stress Test: BTC Price Drop + Withdrawal

In [ ]:
# ─────────────────────────────────────────────
# 7. STRESS TEST: BTC PRICE DROP x WITHDRAWAL
# ─────────────────────────────────────────────
# After a withdrawal, what happens if BTC also drops?
# Post-WD LTV = Loan_Outstanding / (Remaining_BTC_Value * (1 - price_drop))

wd_scenarios = [0.10, 0.25, 0.50, 0.75]
btc_drops    = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

stress_data = []
for wd in wd_scenarios:
    r = results[wd]
    for drop in btc_drops:
        remaining_collateral = r["Collateral Value ($)"] * (1 - drop)
        stressed_ltv = r["Loan Outstanding ($)"] / remaining_collateral if remaining_collateral > 0 else 999
        stress_data.append({
            "Withdrawal %": f"{wd:.0%}",
            "BTC Drop": f"{drop:.0%}",
            "Stressed LTV": stressed_ltv,
            "Remaining Collateral ($M)": remaining_collateral / 1e6,
            "Loan Outstanding ($M)": r["Loan Outstanding ($)"] / 1e6,
        })

stress_df = pd.DataFrame(stress_data)

# Pivot for display
stress_pivot = stress_df.pivot(index="Withdrawal %", columns="BTC Drop", values="Stressed LTV")

def color_ltv(val):
    if val <= 0.65:
        return "background-color: #c6efce; color: #006100"
    elif val <= 0.70:
        return "background-color: #ffeb9c; color: #9c5700"
    elif val <= 0.85:
        return "background-color: #ffc7ce; color: #9c0006"
    else:
        return "background-color: #ff6666; color: white; font-weight: bold"

print("=" * 100)
print("  STRESS TEST: POST-WITHDRAWAL LTV UNDER BTC PRICE DROPS")
print("  Green <= 65% | Yellow 65-70% | Red 70-85% | Dark Red > 85%")
print("=" * 100)

# Format as percentages for text display
stress_display = stress_pivot.copy()
for col in stress_display.columns:
    stress_display[col] = stress_display[col].apply(lambda x: f"{x:.1%}")
print(stress_display.to_string())

# Also show styled version in notebook
stress_pivot.style.applymap(color_ltv).format("{:.1%}")

In [ ]:
# ─────────────────────────────────────────────
# 8. BTC DROP REQUIRED TO HIT MAX LTV (70%)
# ─────────────────────────────────────────────
# For each withdrawal scenario, how much can BTC drop
# before the vault breaches 70% LTV?

print("=" * 65)
print("  BREAK-EVEN BTC PRICE DROP TO BREACH 70% LTV")
print("=" * 65)
print(f"  {'WD %':<12} {'Post-WD LTV':<15} {'Max BTC Drop':<15} {'BTC Price at Breach':<20}")
print("  " + "-" * 60)

BTC_PRICE = INITIAL_AUM / 100  # implied BTC price if 100 BTC

for pct in scenarios:
    r = results[pct]
    ltv = r["Primary LTV After"]
    be_drop = r["Break-even BTC Drop"]
    breach_price = BTC_PRICE * (1 - be_drop) if be_drop > 0 else 0
    status = "ALREADY BREACHED" if be_drop <= 0 else f"${breach_price:,.0f}"
    print(f"  {pct:<12.0%} {ltv:<15.2%} {be_drop:<15.1%} {status:<20}")

---
## P&L Impact Analysis

In [ ]:
# ─────────────────────────────────────────────
# 9. P&L IMPACT — ANNUALIZED YIELD AFTER WITHDRAWAL
# ─────────────────────────────────────────────
pnl_rows = []
for pct in scenarios:
    r = results[pct]
    pnl_rows.append({
        "Withdrawal %"               : f"{pct:.0%}",
        "Remaining AUM ($M)"         : f"${r['Collateral Value ($)']/1e6:.1f}M",
        "Gross RWA Yield ($K)"       : f"${r['Remaining Gross Yield ($)']/1e3:.0f}K",
        "Morpho Cost ($K)"           : f"${r['Remaining Morpho Cost ($)']/1e3:.0f}K",
        "Secondary Cost ($K)"        : f"${r['Annual Secondary Borrow Cost']/1e3:.0f}K",
        "Net Yield ($K)"             : f"${r['Net Yield ($)']/1e3:.0f}K",
        "Net Yield on AUM"           : f"{r['Net Yield on Remaining AUM']:.2%}",
        "Break-even BTC Drop"        : f"{r['Break-even BTC Drop']:.1%}",
    })

pnl_df = pd.DataFrame(pnl_rows)
print("=" * 120)
print("  POST-WITHDRAWAL P&L (ANNUALIZED)")
print("=" * 120)
print(pnl_df.to_string(index=False))

---
## Liquidity Coverage Analysis

In [ ]:
# ─────────────────────────────────────────────
# 10. LIQUIDITY COVERAGE BREAKDOWN
# ─────────────────────────────────────────────
# For each scenario, show how much comes from each waterfall step

coverage_rows = []
for pct in scenarios:
    r = results[pct]
    log = r["Waterfall Log"]
    
    cash_used = sum(l["Amount"] for l in log if l["Step"].startswith("1"))
    short_term = sum(l["Amount"] for l in log if l["Step"].startswith("2"))
    collateral_borrows = sum(l["Amount"] for l in log if l["Step"].startswith("3"))
    total_needed = r["Loan to Repay ($)"]
    
    coverage_rows.append({
        "WD %": pct,
        "Loan to Repay ($M)": total_needed / 1e6,
        "Step 1: Cash ($M)": cash_used / 1e6,
        "Step 2: Short-term ($M)": short_term / 1e6,
        "Step 3: Collateral Borrow ($M)": collateral_borrows / 1e6,
        "Shortfall ($M)": r["Shortfall ($)"] / 1e6,
        "Cash %": cash_used / total_needed if total_needed > 0 else 0,
        "Short-term %": short_term / total_needed if total_needed > 0 else 0,
        "Collateral %": collateral_borrows / total_needed if total_needed > 0 else 0,
    })

cov_df = pd.DataFrame(coverage_rows)

print("=" * 100)
print("  LIQUIDITY SOURCING BREAKDOWN BY WATERFALL STEP")
print("=" * 100)
print(f"  {'WD %':<8} {'Loan ($M)':<12} {'Cash ($M)':<12} {'Short ($M)':<12} {'Borrow ($M)':<14} {'Short ($M)':<12} {'Cash %':<10} {'Short %':<10} {'Borrow %':<10}")
print("  " + "-" * 95)
for _, row in cov_df.iterrows():
    print(f"  {row['WD %']:<8.0%} {row['Loan to Repay ($M)']:<12.2f} {row['Step 1: Cash ($M)']:<12.2f} {row['Step 2: Short-term ($M)']:<12.2f} {row['Step 3: Collateral Borrow ($M)']:<14.3f} {row['Shortfall ($M)']:<12.3f} {row['Cash %']:<10.1%} {row['Short-term %']:<10.1%} {row['Collateral %']:<10.1%}")

---
## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("BTC.B Vault — Withdrawal Waterfall Analysis", fontsize=16, fontweight="bold", y=0.98)

wd_pcts = [pct for pct in scenarios]
wd_labels = [f"{p:.0%}" for p in wd_pcts]

# ── Chart 1: Waterfall Sourcing Stacked Bar ──
ax1 = axes[0, 0]
cash_vals = [sum(l["Amount"] for l in results[p]["Waterfall Log"] if l["Step"].startswith("1")) / 1e6 for p in wd_pcts]
short_vals = [sum(l["Amount"] for l in results[p]["Waterfall Log"] if l["Step"].startswith("2")) / 1e6 for p in wd_pcts]
borrow_vals = [sum(l["Amount"] for l in results[p]["Waterfall Log"] if l["Step"].startswith("3")) / 1e6 for p in wd_pcts]
shortfall_vals = [results[p]["Shortfall ($)"] / 1e6 for p in wd_pcts]

x = np.arange(len(wd_pcts))
w = 0.6
ax1.bar(x, cash_vals, w, label="Step 1: Cash", color="#2e75b6")
ax1.bar(x, short_vals, w, bottom=cash_vals, label="Step 2: Short-term", color="#70ad47")
ax1.bar(x, borrow_vals, w, bottom=[c+s for c,s in zip(cash_vals, short_vals)], label="Step 3: Collateral Borrow", color="#ffc000")
ax1.bar(x, shortfall_vals, w, bottom=[c+s+b for c,s,b in zip(cash_vals, short_vals, borrow_vals)], label="Shortfall", color="#ff4444")

# Overlay: loan to repay line
loan_to_repay_vals = [results[p]["Loan to Repay ($)"] / 1e6 for p in wd_pcts]
ax1.plot(x, loan_to_repay_vals, 'k--o', label="Loan to Repay", linewidth=2, markersize=5)

ax1.set_xticks(x)
ax1.set_xticklabels(wd_labels)
ax1.set_xlabel("Withdrawal %")
ax1.set_ylabel("USD (Millions)")
ax1.set_title("Liquidity Sourcing by Waterfall Step")
ax1.legend(fontsize=8, loc="upper left")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.0f}M"))

# ── Chart 2: Post-Withdrawal LTV ──
ax2 = axes[0, 1]
ltvs = [results[p]["Primary LTV After"] for p in wd_pcts]
colors = ["#c6efce" if l <= 0.65 else "#ffeb9c" if l <= 0.70 else "#ff4444" for l in ltvs]
bars = ax2.bar(x, [l * 100 for l in ltvs], w, color=colors, edgecolor="gray")
ax2.axhline(y=65, color="orange", linestyle="--", label="Target LTV (65%)")
ax2.axhline(y=70, color="red", linestyle="--", linewidth=2, label="Max LTV (70%)")
ax2.set_xticks(x)
ax2.set_xticklabels(wd_labels)
ax2.set_xlabel("Withdrawal %")
ax2.set_ylabel("LTV (%)")
ax2.set_title("Post-Withdrawal Primary LTV")
ax2.set_ylim(0, 80)
ax2.legend(fontsize=9)
for bar, ltv in zip(bars, ltvs):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f"{ltv:.1%}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Chart 3: Net Yield vs Withdrawal ──
ax3 = axes[1, 0]
net_yields_pct = [results[p]["Net Yield on Remaining AUM"] * 100 for p in wd_pcts]
gross_yields = [results[p]["Remaining Gross Yield ($)"] / results[p]["Collateral Value ($)"] * 100 if results[p]["Collateral Value ($)"] > 0 else 0 for p in wd_pcts]

ax3.plot(wd_labels, gross_yields, 'g-o', label="Gross RWA Yield (% AUM)", linewidth=2)
ax3.plot(wd_labels, net_yields_pct, 'b-s', label="Net Yield (% AUM)", linewidth=2)
ax3.axhline(y=0, color="red", linestyle=":", alpha=0.5)
ax3.fill_between(range(len(wd_pcts)), net_yields_pct, alpha=0.1, color="blue")
ax3.set_xlabel("Withdrawal %")
ax3.set_ylabel("Annualized Yield (% of Remaining AUM)")
ax3.set_title("P&L Impact: Yield Erosion from Withdrawals")
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# ── Chart 4: Break-even BTC Drop ──
ax4 = axes[1, 1]
be_drops = [results[p]["Break-even BTC Drop"] * 100 for p in wd_pcts]
colors4 = ["#c6efce" if d > 5 else "#ffeb9c" if d > 2 else "#ff4444" for d in be_drops]
bars4 = ax4.bar(x, be_drops, w, color=colors4, edgecolor="gray")
ax4.set_xticks(x)
ax4.set_xticklabels(wd_labels)
ax4.set_xlabel("Withdrawal %")
ax4.set_ylabel("BTC Price Drop (%)")
ax4.set_title("Break-even: BTC Drop to Breach 70% LTV")
ax4.axhline(y=5, color="orange", linestyle="--", alpha=0.7, label="5% threshold")
ax4.legend(fontsize=9)
for bar, d in zip(bars4, be_drops):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
             f"{d:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("/Users/adrianchow/Desktop/btc_vault_waterfall_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved to Desktop.")

In [ ]:
# ─────────────────────────────────────────────
# 11. STRESS HEATMAP
# ─────────────────────────────────────────────
import matplotlib.colors as mcolors

fig2, ax = plt.subplots(figsize=(12, 5))

# Build stress matrix
wd_stress = [0.05, 0.10, 0.25, 0.50, 0.75]
btc_drops_hm = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

# Run waterfall for any missing scenarios
for wd in wd_stress:
    if wd not in results:
        results[wd] = run_waterfall(wd, assets)

heatmap_data = []
for wd in wd_stress:
    row = []
    for drop in btc_drops_hm:
        r = results[wd]
        remaining_coll = r["Collateral Value ($)"] * (1 - drop)
        ltv = r["Loan Outstanding ($)"] / remaining_coll if remaining_coll > 0 else 1.5
        row.append(ltv)
    heatmap_data.append(row)

heatmap_arr = np.array(heatmap_data)

# Custom colormap: green -> yellow -> red -> dark red
cmap = mcolors.LinearSegmentedColormap.from_list(
    "ltv", [(0, "#c6efce"), (0.55/1.2, "#c6efce"), (0.65/1.2, "#ffeb9c"),
            (0.70/1.2, "#ffc7ce"), (0.85/1.2, "#ff4444"), (1.0, "#990000")])

im = ax.imshow(heatmap_arr, cmap=cmap, aspect="auto", vmin=0.50, vmax=1.20)

# Labels
ax.set_xticks(range(len(btc_drops_hm)))
ax.set_xticklabels([f"{d:.0%}" for d in btc_drops_hm])
ax.set_yticks(range(len(wd_stress)))
ax.set_yticklabels([f"{w:.0%}" for w in wd_stress])
ax.set_xlabel("BTC Price Drop", fontsize=12)
ax.set_ylabel("Withdrawal %", fontsize=12)
ax.set_title("Stress Test Heatmap: Post-Event Morpho LTV", fontsize=14, fontweight="bold")

# Annotate cells
for i in range(len(wd_stress)):
    for j in range(len(btc_drops_hm)):
        val = heatmap_arr[i, j]
        color = "white" if val > 0.80 else "black"
        ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=9, color=color, fontweight="bold")

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("LTV", fontsize=11)

# Add threshold lines annotation
ax.text(len(btc_drops_hm) + 0.5, -0.8, "Target: 65% | Max: 70%", fontsize=10, color="red", fontweight="bold")

plt.tight_layout()
plt.savefig("/Users/adrianchow/Desktop/btc_vault_stress_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Stress heatmap saved to Desktop.")

---
## Key Findings & Risk Summary

In [ ]:
# ─────────────────────────────────────────────
# 12. KEY FINDINGS
# ─────────────────────────────────────────────
print("=" * 70)
print("  KEY FINDINGS")
print("=" * 70)

# Find the max withdrawal that stays healthy
max_healthy = max([p for p in scenarios if results[p]["Primary LTV Healthy"]], default=0)
max_sla     = max([p for p in scenarios if results[p]["50% SLA Met"]], default=0)
first_shortfall = min([p for p in scenarios if results[p]["Shortfall ($)"] > 0], default=None)

print(f"\n  1. LTV HEALTH")
print(f"     - Max withdrawal with LTV <= 70%: {max_healthy:.0%}")
print(f"     - At 65% target LTV, the waterfall fully repays proportional")
print(f"       loan for all scenarios up to {max_healthy:.0%} of AUM")

print(f"\n  2. SLA COMPLIANCE")
print(f"     - 50% SLA met for withdrawals up to: {max_sla:.0%}")
print(f"     - Cash + short-term assets (<=7d) = 65% of portfolio")
print(f"     - With collateral borrowing: near-100% 7d coverage")

print(f"\n  3. LIQUIDITY CONSTRAINTS")
if first_shortfall:
    print(f"     - First shortfall appears at {first_shortfall:.0%} withdrawal")
    print(f"     - Shortfall: ${results[first_shortfall]['Shortfall ($)']/1e6:.3f}M")
else:
    print(f"     - No liquidity shortfall in any scenario up to 75%")

print(f"\n  4. SECONDARY BORROWING EXPOSURE")
for pct in [0.50, 0.75]:
    r = results[pct]
    if r["Secondary Borrows ($)"] > 0:
        print(f"     - {pct:.0%} WD: ${r['Secondary Borrows ($)']/1e6:.2f}M borrowed against RWA")
        print(f"       Annual cost: ${r['Annual Secondary Borrow Cost']/1e3:.0f}K")

print(f"\n  5. BTC PRICE SENSITIVITY")
print(f"     - At 65% LTV, only a {results[0.05]['Break-even BTC Drop']:.1%} BTC drop breaches 70%")
print(f"     - This is the primary risk factor regardless of withdrawal size")
print(f"     - Combined stress (50% WD + 20% BTC drop):")
r50 = results[0.50]
stressed_ltv = r50["Loan Outstanding ($)"] / (r50["Collateral Value ($)"] * 0.80)
print(f"       LTV = {stressed_ltv:.1%}")